<a href="https://www.kaggle.com/code/tommasofacchin/01-resident-evil-data-scraping?scriptVersionId=304849559" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np 
import pandas as pd 
import requests
from pathlib import Path
from bs4 import BeautifulSoup

# Datesets

**Game** (id, title, year, type)

**Character** (id, name, role, faction)

**GameAppearence** (gameId, characterId, role)

**Dialogues** (id, game_id, scene_id, line_index, character_id, text, timestamp, source)

**Scene** (id, game_id, name, sequence_index)

In [2]:
games_data = [
    # id, title, year, type, chronology_order
    (1,  "Resident Evil",                             1996, "mainline", 2),
    (2,  "Resident Evil 2",                           1998, "mainline", 4),
    (3,  "Resident Evil 3: Nemesis",                  1999, "mainline", 3),
    (4,  "Resident Evil: Code – Veronica",            2000, "mainline", 5),
    (5,  "Resident Evil 0",                           2002, "mainline", 1),
    (6,  "Resident Evil 4",                           2005, "mainline", 6),
    (7,  "Resident Evil 5",                           2009, "mainline", 8),
    (8,  "Resident Evil: Revelations",                2012, "spinoff",  7),
    (9,  "Resident Evil 6",                           2012, "mainline", 10),
    (10, "Resident Evil: Revelations 2",              2015, "spinoff",  9),
    (11, "Resident Evil 7: Biohazard",                2017, "mainline", 11),
    (12, "Resident Evil Village",                     2021, "mainline", 12),
    (13, "Resident Evil Village – Shadow of Rose",    2022, "spinoff",  13),
    (14, "Resident Evil Requiem",                     2026, "mainline", 14),
]

games = pd.DataFrame(
    games_data,
    columns=["id", "title", "year", "type", "chronology_order"]
)

games.to_csv("games.csv", index=False)
games.head(20)

,id,title,year,type,chronology_order
0,1,Resident Evil,1996,mainline,2
1,2,Resident Evil 2,1998,mainline,4
2,3,Resident Evil 3: Nemesis,1999,mainline,3
3,4,Resident Evil: Code – Veronica,2000,mainline,5
4,5,Resident Evil 0,2002,mainline,1
5,6,Resident Evil 4,2005,mainline,6
6,7,Resident Evil 5,2009,mainline,8
7,8,Resident Evil: Revelations,2012,spinoff,7
8,9,Resident Evil 6,2012,mainline,10
9,10,Resident Evil: Revelations 2,2015,spinoff,9


In [3]:
#Italian website has more characters
URL = "https://it.wikipedia.org/wiki/Personaggi_di_Resident_Evil"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

resp = requests.get(URL, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

# Because kaggle run notebook offline
HTML_PATH = "/kaggle/input/datasets/tommasofacchin/personaggi-resident-evil-wikipedia/personaggi_resident_evil_it.html"
with open(HTML_PATH, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")
# ---


section_role = {
    "Personaggi principali": "hero",
    "Antagonisti principali": "villain",
    "Personaggi secondari": "support",
    "Antagonisti secondari": "villain",
}

names = []
roles = []

current_role = None

for tag in soup.find_all(["h2", "h3"]):
    if tag.name == "h2":
        title = tag.get_text(strip=True)
        current_role = None
        for sec, role in section_role.items():
            if title.startswith(sec):
                current_role = role
                break

    elif tag.name == "h3" and current_role is not None:
        name = tag.get_text(strip=True)
        names.append(name)
        roles.append(current_role)


heroes = [n for n, r in zip(names, roles) if r == "hero"]
villains = [n for n, r in zip(names, roles) if r == "villain"]
supports = [n for n, r in zip(names, roles) if r == "support"]

print("\n# hero:", len(heroes))
print(heroes)

print("\n# villain:", len(villains))
print(villains)

print("\n# support:", len(supports))
print(supports)


# hero: 9
['Chris Redfield', 'Leon Scott Kennedy', 'Jill Valentine', 'Claire Redfield', 'Ada Wong', 'Ethan Winters', 'Grace Ashcroft', 'Rebecca Chambers', 'S.T.A.R.S.']

# villain: 12
['Albert Wesker', 'William Birkin', 'Lord Osmund Saddler', 'James Marcus', 'Sergei Vladimir', 'Alfred Ashford', 'Alexia Ashford', 'Derek C. Simmons', 'Ozwell E. Spencer', 'Ramon Salazar', 'Jack Krauser', 'Bitores Mendez']

# support: 14
['Barry Burton', 'Sherry Birkin', 'Carlos Oliveira', 'Ashley Graham', 'Luis Sera Navarro', 'Billy Coen', 'Ingrid Hunnigan', 'Helena Harper', 'Debora Harper', 'Jake Muller', 'Piers Nivans', 'Parker Luciani', 'Moira Burton', 'Lucia']


In [4]:
characters = pd.DataFrame({"name": names, "role": roles})
characters.insert(0, "id", range(1, len(characters) + 1))

characters.to_csv("characters.csv", index=False)
characters.head()

,id,name,role
0,1,Chris Redfield,hero
1,2,Leon Scott Kennedy,hero
2,3,Jill Valentine,hero
3,4,Claire Redfield,hero
4,5,Ada Wong,hero


In [5]:
characters = pd.DataFrame(
    columns=[
        "id",       
        "name",    
        "role",     
        #"faction",  
    ]
)

gameAppearance = pd.DataFrame(
    columns=[
        "gameId",       
        "characterId", 
        "role",        
    ]
)

In [6]:
game_id = 6
URL = "https://it.wikipedia.org/wiki/Resident_Evil_4"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

resp = requests.get(URL, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

names = []
roles = [] 
current_role = None 

for tag in soup.find_all(["h2", "h3"]):
    if tag.name == "h2":
        text = tag.get_text(strip=True)
        print("H2:", text)  

        if text == "Personaggi":
            current_role = "hero"
        elif text == "Nemici":
            current_role = "villain"
        else:
            current_role = None

    elif tag.name == "h3" and current_role is not None:
        text = tag.get_text(strip=True)

        names.append(text)
        roles.append(current_role)
        print("  H3:", text)

print("\n# characters:", len(names))
print(list(zip(names, roles)))


re4_df = pd.DataFrame({
    "name": names,
    "role_in_game": roles,  
})

# Characters
existing_names = set(characters["name"])

missing_mask = ~re4_df["name"].isin(existing_names)
missing_rows = re4_df[missing_mask]

next_id = characters["id"].max() + 1 if len(characters) > 0 else 1

new_rows = []
for _, row in missing_rows.iterrows():
    new_rows.append({
        "id": next_id,
        "name": row["name"],
        "role": row["role_in_game"],
    })
    next_id += 1

if new_rows:
    new_chars = pd.DataFrame(new_rows)
    characters = pd.concat([characters, new_chars], ignore_index=True)

# Game appearences
rows = []
for _, row in re4_df.iterrows():
    name = row["name"]
    role_in_game = row["role_in_game"]

    match = characters[characters["name"] == name]
    if match.empty:
        continue
    cid = match["id"].iloc[0]

    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re4 = pd.DataFrame(rows)

gameAppearance = pd.concat([gameAppearance, gameApp_re4], ignore_index=True)

H2: Indice
H2: Trama
H2: Modalità di gioco
H2: Personaggi
  H3: Leon Scott Kennedy
  H3: Ashley Graham
  H3: Ada Wong
  H3: Ingrid Hunnigan
  H3: Luis Sera
  H3: Albert Wesker
  H3: HUNK
  H3: Mercante d'armi
H2: Nemici
  H3: Las Plagas
  H3: Los Ganados
  H3: Lord Osmund Saddler
  H3: Ramon Salazar
  H3: Bitores Mendez
  H3: Jack Krauser
H2: Versioni
H2: Confronti territoriali
H2: Versione beta
H2: Remake
H2: Edizione VR
H2: Accoglienza
H2: Doppiaggio
H2: Colonna sonora
H2: Note
H2: Altri progetti
H2: Collegamenti esterni

# characters: 14
[('Leon Scott Kennedy', 'hero'), ('Ashley Graham', 'hero'), ('Ada Wong', 'hero'), ('Ingrid Hunnigan', 'hero'), ('Luis Sera', 'hero'), ('Albert Wesker', 'hero'), ('HUNK', 'hero'), ("Mercante d'armi", 'hero'), ('Las Plagas', 'villain'), ('Los Ganados', 'villain'), ('Lord Osmund Saddler', 'villain'), ('Ramon Salazar', 'villain'), ('Bitores Mendez', 'villain'), ('Jack Krauser', 'villain')]


In [7]:
characters.head(50)

,id,name,role
0,1,Leon Scott Kennedy,hero
1,2,Ashley Graham,hero
2,3,Ada Wong,hero
3,4,Ingrid Hunnigan,hero
4,5,Luis Sera,hero
5,6,Albert Wesker,hero
6,7,HUNK,hero
7,8,Mercante d'armi,hero
8,9,Las Plagas,villain
9,10,Los Ganados,villain


In [8]:
gameAppearance.head(50)

,gameId,characterId,role
0,6,1,hero
1,6,2,hero
2,6,3,hero
3,6,4,hero
4,6,5,hero
5,6,6,hero
6,6,7,hero
7,6,8,hero
8,6,9,villain
9,6,10,villain
